In [2]:
# Imports
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load and merge data
trans = pd.read_csv('../data/raw/train_transaction.csv')
ident = pd.read_csv('../data/raw/train_identity.csv')

df = trans.merge(ident, on='TransactionID', how='left')
df = df.copy()

print(f"Shape: {df.shape}")
print("Data loaded successfully")

Shape: (590540, 434)
Data loaded successfully


In [4]:
# Time-based train/validation/test split
TRAIN_CUTOFF = 10_438_003
VAL_CUTOFF   = 13_151_880

df_sorted = df.sort_values('TransactionDT').reset_index(drop=True)

train = df_sorted[df_sorted['TransactionDT'] < TRAIN_CUTOFF].copy()
val   = df_sorted[(df_sorted['TransactionDT'] >= TRAIN_CUTOFF) & 
                  (df_sorted['TransactionDT'] < VAL_CUTOFF)].copy()
test  = df_sorted[df_sorted['TransactionDT'] >= VAL_CUTOFF].copy()

print(f"Train: {len(train):,} rows | fraud rate: {train['isFraud'].mean():.4f}")
print(f"Val:   {len(val):,} rows | fraud rate: {val['isFraud'].mean():.4f}")
print(f"Test:  {len(test):,} rows | fraud rate: {test['isFraud'].mean():.4f}")
print(f"Total: {len(train)+len(val)+len(test):,} rows")

# Separate features and target
X_train = train.drop(columns=['isFraud', 'TransactionID'])
y_train = train['isFraud']

X_val = val.drop(columns=['isFraud', 'TransactionID'])
y_val = val['isFraud']

X_test = test.drop(columns=['isFraud', 'TransactionID'])
y_test = test['isFraud']

print(f"\nX_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")

Train: 413,378 rows | fraud rate: 0.0352
Val:   88,581 rows | fraud rate: 0.0343
Test:  88,581 rows | fraud rate: 0.0348
Total: 590,540 rows

X_train shape: (413378, 432)
X_val shape:   (88581, 432)
X_test shape:  (88581, 432)


In [5]:
# Datetime transformer
class DatetimeFeatures(BaseEstimator, TransformerMixin):
    """
    Extracts temporal features from TransactionDT.
    TransactionDT is a seconds offset from an unknown reference point.
    No fitting required, pure arithmetic transformation.
    """
    
    def fit(self, X, y=None):
        return self  # nothing to learn
    
    def transform(self, X):
        X = X.copy()
        X['hour_of_day']      = (X['TransactionDT'] // 3600) % 24
        # day_of_week: 7-day cycle derived from TransactionDT offset
        # Day 0 does not correspond to a known weekday, relative cycle only
        X['day_of_week']      = (X['TransactionDT'] // 86400) % 7
        X['is_weekend']       = (X['day_of_week'] >= 5).astype(int)
        X['time_since_start'] = X['TransactionDT']
        return X

# Test it
dt = DatetimeFeatures()
X_train_test = dt.fit_transform(X_train)

print("New columns added:")
print(X_train_test[['hour_of_day','day_of_week','is_weekend','time_since_start']].head())
print(f"\nShape before: {X_train.shape} | after: {X_train_test.shape}")

New columns added:
   hour_of_day  day_of_week  is_weekend  time_since_start
0            0            1           0             86400
1            0            1           0             86401
2            0            1           0             86469
3            0            1           0             86499
4            0            1           0             86506

Shape before: (413378, 432) | after: (413378, 436)


In [6]:
# Cell 5 — Amount features transformer
class AmountFeatures(BaseEstimator, TransformerMixin):
    """
    Derives features from TransactionAmt.
    No fitting required — pure arithmetic transformation.
    Log transform handles right skew (skewness=14.37 found in EDA).
    """

    def fit(self, X, y=None):
        return self  # nothing to learn

    def transform(self, X):
        X = X.copy()
        X['amt_log']     = np.log1p(X['TransactionAmt'])
        X['amt_rounded'] = (X['TransactionAmt'] % 1 == 0).astype(int)
        X['amt_cents']   = X['TransactionAmt'] % 1
        return X

# Test it
amt = AmountFeatures()
X_train_test = amt.fit_transform(X_train)

print("New columns added:")
print(X_train_test[['TransactionAmt','amt_log','amt_rounded','amt_cents']].head())
print(f"\nShape before: {X_train.shape} | after: {X_train_test.shape}")

New columns added:
   TransactionAmt   amt_log  amt_rounded  amt_cents
0            68.5  4.241327            0        0.5
1            29.0  3.401197            1        0.0
2            59.0  4.094345            1        0.0
3            50.0  3.931826            1        0.0
4            50.0  3.931826            1        0.0

Shape before: (413378, 432) | after: (413378, 435)


In [7]:
# Cell 6 — Email features transformer
class EmailFeatures(BaseEstimator, TransformerMixin):
    """
    Frequency encodes email domains and adds match flag.
    fit() learns domain frequencies from training data only.
    transform() maps any split using those learned frequencies.
    NaN domains treated as a separate category ('missing').
    """

    def fit(self, X, y=None):
        # Learn frequency of each domain from training data
        self.p_email_freq_ = X['P_emaildomain'].fillna('missing').value_counts(normalize=True)
        self.r_email_freq_ = X['R_emaildomain'].fillna('missing').value_counts(normalize=True)
        return self

    def transform(self, X):
        X = X.copy()
        p_filled = X['P_emaildomain'].fillna('missing')
        r_filled = X['R_emaildomain'].fillna('missing')

        # Map frequencies — unseen domains in val/test get 0
        X['P_email_freq'] = p_filled.map(self.p_email_freq_).fillna(0)
        X['R_email_freq'] = r_filled.map(self.r_email_freq_).fillna(0)
        X['email_match']  = (p_filled == r_filled).astype(int)
        return X

# Test it
email = EmailFeatures()
email.fit(X_train)
X_train_test = email.transform(X_train)

print("New columns added:")
print(X_train_test[['P_emaildomain','P_email_freq','R_emaildomain','R_email_freq','email_match']].head(8))
print(f"\nShape before: {X_train.shape} | after: {X_train_test.shape}")

New columns added:
  P_emaildomain  P_email_freq R_emaildomain  R_email_freq  email_match
0           NaN      0.155538           NaN      0.750734            1
1     gmail.com      0.385154           NaN      0.750734            0
2   outlook.com      0.008588           NaN      0.750734            0
3     yahoo.com      0.170314           NaN      0.750734            0
4     gmail.com      0.385154           NaN      0.750734            0
5     gmail.com      0.385154           NaN      0.750734            0
6     yahoo.com      0.170314           NaN      0.750734            0
7      mail.com      0.000963           NaN      0.750734            0

Shape before: (413378, 432) | after: (413378, 435)


In [8]:
# Cell 7 — Card aggregation features transformer
class CardAggFeatures(BaseEstimator, TransformerMixin):
    """
    Per-card behavioral aggregations using standard mean/std/count.
    fit() computes statistics from training data only.
    transform() maps any split using learned statistics.
    Unseen cards fall back to global training statistics.

    Known limitation: within-training leakage exists because each
    transaction contributes to its own card's mean. Impact is negligible
    at this dataset scale (~30 transactions per card on average).
    LOO encoding is documented as a future improvement.
    """

    def fit(self, X, y=None):
        stats = X.groupby('card1')['TransactionAmt'].agg(['mean','std','count'])
        self.card1_mean_   = stats['mean']
        self.card1_std_    = stats['std']
        self.card1_count_  = stats['count']
        self.global_mean_  = X['TransactionAmt'].mean()
        self.global_std_   = X['TransactionAmt'].std()
        return self

    def transform(self, X):
        X = X.copy()
        X['card1_amt_mean'] = X['card1'].map(self.card1_mean_).fillna(self.global_mean_)
        X['card1_amt_std']  = X['card1'].map(self.card1_std_).fillna(self.global_std_)
        X['card1_tx_count'] = X['card1'].map(self.card1_count_).fillna(1.0)
        return X

# Test on both train and val
card_agg = CardAggFeatures()
card_agg.fit(X_train)
X_train_test = card_agg.transform(X_train)
X_val_test   = card_agg.transform(X_val)

print("New columns added:")
print(X_train_test[['card1','card1_amt_mean','card1_amt_std','card1_tx_count']].head(8))
print(f"\nTrain shape: {X_train.shape} → {X_train_test.shape}")
print(f"Val shape:   {X_val.shape} → {X_val_test.shape}")

New columns added:
   card1  card1_amt_mean  card1_amt_std  card1_tx_count
0  13926      367.688929     383.506436              28
1   2755      244.516341     501.969085             522
2   4663       96.776658     102.847409             769
3  18132      122.976838     187.744406            2944
4   4497      105.083333      55.003125               9
5   5937      148.250000     101.257963               6
6  12308      107.889438     196.564359             160
7  12695      141.476273     206.269243            4772

Train shape: (413378, 432) → (413378, 435)
Val shape:   (88581, 432) → (88581, 435)


In [9]:
# Cell 8 — M features transformer (your version, corrected)
class MFeatures(BaseEstimator, TransformerMixin):
    """
    Encodes M1-M9 match flag features.
    M1-M3, M5-M9: three-state encoding T=1, F=0, NaN=-1
    M4: one-hot encoding with fixed columns learned during fit()
        Unseen categories in val/test dropped.
        Missing expected columns filled with 0.
    Adds M_sum_true and M_sum_false as behavioral aggregates.
    """

    def fit(self, X, y=None):
        self.binary_m = ['M1','M2','M3','M5','M6','M7','M8','M9']
        cats = set(X['M4'].dropna().unique())
        cats.add('missing')
        self.m4_categories_ = sorted([f"M4_{cat}" for cat in cats])
        return self

    def transform(self, X):
        X = X.copy()

        # Vectorized three-state encoding
        X[self.binary_m] = (
            X[self.binary_m]
            .replace({'T': 1, 'F': 0})
            .fillna(-1)
        )

        # One-hot encoding for M4 — enforce training columns
        m4_dummies = pd.get_dummies(
            X['M4'].fillna('missing'), prefix='M4'
        ).astype(int)

        m4_dummies = m4_dummies.reindex(
            columns=self.m4_categories_,
            fill_value=0
        )

        # Behavioral aggregates
        X['M_sum_true']  = (X[self.binary_m] == 1).sum(axis=1)
        X['M_sum_false'] = (X[self.binary_m] == 0).sum(axis=1)

        X = pd.concat([X.drop(columns=['M4']), m4_dummies], axis=1)
        return X

# Test on both train and val
m_feat = MFeatures()
m_feat.fit(X_train)
X_train_test = m_feat.transform(X_train)
X_val_test   = m_feat.transform(X_val)

print("M4 columns learned during fit:")
print(m_feat.m4_categories_)
print(X_train_test[['M1','M2','M3','M_sum_true','M_sum_false'] + m_feat.m4_categories_].head(8))
print(f"\nTrain shape: {X_train.shape} → {X_train_test.shape}")
print(f"Val shape:   {X_val.shape} → {X_val_test.shape}")

M4 columns learned during fit:
['M4_M0', 'M4_M1', 'M4_M2', 'M4_missing']
   M1  M2  M3  M_sum_true  M_sum_false  M4_M0  M4_M1  M4_M2  M4_missing
0   1   1   1           4            1      0      0      1           0
1  -1  -1  -1           2            0      1      0      0           0
2   1   1   1           3            5      1      0      0           0
3  -1  -1  -1           1            1      1      0      0           0
4  -1  -1  -1           0            0      0      0      0           1
5   1   1   1           4            1      0      1      0           0
6   1   1   1           6            2      1      0      0           0
7  -1  -1  -1           0            2      1      0      0           0

Train shape: (413378, 432) → (413378, 437)
Val shape:   (88581, 432) → (88581, 437)


In [10]:
class DFeatures(BaseEstimator, TransformerMixin):
    """
    Handles D1-D15 time delta features.
    Missing values imputed with -1 (sentinel — impossible natural value).
    Binary is_missing flag added per column.
    No fitting required.
    """

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        d_cols = [f'D{i}' for i in range(1, 16)]
        for col in d_cols:
            X[f'{col}_missing'] = X[col].isna().astype(int)
            X[col] = X[col].fillna(-1)
        return X

# Test it
d_feat = DFeatures()
X_train_test = d_feat.fit_transform(X_train)

print("D1 sample:")
print(X_train_test[['D1', 'D1_missing', 'D2', 'D2_missing']].head(8))
print(f"\nShape before: {X_train.shape} | after: {X_train_test.shape}")


D1 sample:
      D1  D1_missing     D2  D2_missing
0   14.0           0   -1.0           1
1    0.0           0   -1.0           1
2    0.0           0   -1.0           1
3  112.0           0  112.0           0
4    0.0           0   -1.0           1
5    0.0           0   -1.0           1
6    0.0           0   -1.0           1
7    0.0           0   -1.0           1

Shape before: (413378, 432) | after: (413378, 447)
